# PT-W4-D7 实验 · Digital Employee Validation：跑验证、出报告

D6 设计了验证（夹具 / 判定梯 / 判分表），今天**跑**。但跑验证 ≠ 得到"通过"两个字，产出是一份**七段结构验证报告**：范围 / 判定结果 / 对照证据 / 缺陷账本 / 覆盖声明 / 闸门裁决 / v0.2 待办。

今天加两个 D6 没有定式化的动作：
1. **对照基线**——引入一个故意做错的"快照式 Agent"（信任落库 `available` 字段，不推理）。基线若也通过，说明验证本身没有牙齿；
2. **盲测**——加一个设计时未讨论的新夹具 A104，检验 Rule Card 声明的泛化力。

In [ ]:
# matplotlib 中文字体配置（TOOLS.md 唯一标准方式）
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 1. 两名参赛 Agent + 四个夹具

- **快照式基线**：直接读落库 `snapshot_available` 字段回答（D-001 Amendment A 反对、但行业主流实际在做的方式）
- **语义模型 Agent**：六层模型推理——从事实重算（CRE-R-002 Derivation），每跳携带 evidence ID

夹具世界状态（A101-A103 来自 D6 设计，A104 为盲测新增）：

In [ ]:
from datetime import date
TODAY = date(2026, 8, 23)   # 固定"今天"，保证报告可复现

# ── D3 Rule Card（精简：谓词结构化声明，非为夹具定制的 if-else）──
RULES = {
    "CRE-R-002": dict(species="Derivation",
        stmt="AvailableForLeasing = Asset.Active ∧ no Occupancy ∧ no Restriction",
        preds=["Space.asset_status == Active", "Space.restriction_flag == absent",
               "Lease.state notin Active,Terminating", "Reservation.state notin Reserved"]),
    "CRE-R-003": dict(species="Guard", mount="T2: Terminating→Terminated",
        stmt="终止迁移前：Inspection 完成 ∧ 清算完成"),
    "CRE-R-008": dict(species="Guard", mount="T3: Reserved→Released",
        stmt="预定到期自动释放；生效期间他人不可选"),
}
# ── D2 迁移（effect-registry 冻结 5 类中的引用）──
TRANSITIONS = {"T2": ("Terminating","Terminated", ["occupancy","financial"]),
               "T3": ("Reserved","Released", ["occupancy"])}

def fixture(sid, path, asset, restriction, occ=None, res=None, snap=None):
    return dict(sid=sid, path=path, asset_status=asset, restriction_flag=restriction,
                occ=occ, res=res, snap_available=snap)

WORLDS = {
    "A101 反例":  fixture("A101", ("万达广场-XX店","1号楼","F1","A101"), "Active","absent",
        occ=dict(lease="L-2024-088", state="Terminating", inspection="pending", settlement="completed"),
        snap=False),                                                       # 快照碰巧对
    "A102 正例":  fixture("A102", ("万达广场-XX店","1号楼","F1","A102"), "Active","absent",
        occ=None, res=None, snap=True),                                    # 快照碰巧对
    "A103 边界例": fixture("A103", ("万达广场-XX店","2号楼","B1","A103"), "Active","absent",
        res=dict(rid="R-0917", state="Reserved", expiry=date(2026,8,20)),
        snap=False),                                                      # ← 快照停留在预定期间，无人刷新
    "A104 盲测":  fixture("A104", ("中旅广场-YY店","1号楼","F2","A204"), "Active","资质审查中",
        occ=None, res=None, snap=True),                                    # ← 盲测：限制场景，快照 stale
}
EXPECTED = {"A101 反例": False, "A102 正例": True, "A103 边界例": True, "A104 盲测": False}  # 预期可租与否

for k, w in WORLDS.items():
    occ = (w["occ"]["lease"] + "/" + w["occ"]["state"]) if w["occ"] else "无"
    res = (w["res"]["rid"] + " 至" + str(w["res"]["expiry"])) if w["res"] else "无"
    print(f"{k}: {w['sid']} | occ: {occ} | res: {res} | 限制: {w['restriction_flag']} | 落库快照 available={w['snap_available']}")

## 2. 语义模型 Agent：每跳带 evidence 的 L1→L5 轨迹

推理核心：**从事实重算，不信任快照**（Derivation）。可租性 = CRE-R-002 四个谓词现场求值。

In [ ]:
def semantic_agent(w):
    """语义模型 Agent：六层推理，返回 L1→L5 轨迹 + 最终答案"""
    trace = {}
    trace["L1"] = ("pass", "身份路径 " + " → ".join(w["path"]), "D1 Identity")
    # L2 业务链：沿 occupies/holds 遍历 + 状态机节点定位
    if w["occ"]:
        trace["L2"] = ("pass", f"occupies ← Lease {w['occ']['lease']}，节点={w['occ']['state']}（还差 T2 才释放）", "T2")
    elif w["res"]:
        node = "Released（T3 已触发）" if w["res"]["expiry"] < TODAY else "Reserved（生效中）"
        trace["L2"] = ("pass", f"holds ← Reservation {w['res']['rid']}，节点={node}", "T3 + CRE-R-008")
    else:
        trace["L2"] = ("pass", "无 occupancy / reservation，Vacant", "D2 状态机")
    # L3 规则判断：CRE-R-002 从事实重算（Derivation）
    if w["asset_status"] != "Active":
        trace["L3"], ans = ("pass", "False（资产状态非 Active）", "CRE-R-002#1"), False
    elif w["restriction_flag"] != "absent":
        trace["L3"], ans = ("pass", f"False（限制未解除：{w['restriction_flag']}）", "CRE-R-002#2"), False
    elif w["occ"] and w["occ"]["state"] in ("Active","Terminating"):
        if w["occ"]["state"] == "Terminating":
            failed = [n for n,v in (("Inspection",w["occ"]["inspection"]),("清算",w["occ"]["settlement"])) if v!="completed"]
            trace["L3"] = ("pass", f"False：存在未完成退租流程（守卫未满足：{'、'.join(failed)}）", "CRE-R-002#3 → CRE-R-003@T2")
        else:
            trace["L3"] = ("pass", "False：存在 Active Occupancy", "CRE-R-002#3 + CRE-R-001")
        ans = False
    elif w["res"] and w["res"]["expiry"] >= TODAY:
        trace["L3"], ans = ("pass", "False：预定生效中，他人不可选", "CRE-R-002#4 + CRE-R-008"), False
    else:
        trace["L3"], ans = ("pass", "True：Active ∧ 无占用 ∧ 无限制" + ("（预定已过期释放）" if w["res"] else ""), "CRE-R-002"), True
    # L4 Policy 判断 / L5 动作建议
    if w["occ"] and w["occ"]["state"] == "Terminating":
        trace["L4"] = ("pass", "创建验收任务免审批（终止申请已过审批；巡检创建走②层 conditional 授权）", "Policy② conditional")
        trace["L5"] = ("pass", "建议 ops.inspection.create：conditional_write + human_review_gate，运营数字员工边界内",
                       "ops.inspection.create + AgentCard[运营]")
    else:
        trace["L4"] = ("n/a", "不适用", "—"); trace["L5"] = ("n/a", "不适用", "—")
    return trace, ans

def snapshot_agent(w):
    """快照式基线：信任落库字段，零推理"""
    return {"L1-L5": ("none", f"available={w['snap_available']}（落库快照，无人刷新）", "无 evidence")}, w["snap_available"]

# ── 正跑：A101 全轨迹展示 ──
trace, ans = semantic_agent(WORLDS["A101 反例"])
print("━━━ A101 正跑 · 语义模型 Agent 轨迹（验证对象是轨迹，不是答案）━━━")
for lv, (v, desc, ev) in trace.items():
    print(f"  {lv} [{v:>4}] {desc}\n        evidence: {ev}")
print(f"  答案: {'可租' if ans else '不可租'}（预期 {'可租' if EXPECTED['A101 反例'] else '不可租'}）")

## 3. 对照赛 + 盲测：验证的判别力与泛化力

四夹具 × 两 Agent。判别力 = 基线答错而语义 Agent 答对的夹具数——**基线全对 = 验证无牙齿**。

In [ ]:
print(f"{'夹具':<12}{'基线答':>8}{'语义答':>8}{'预期':>8}   判别")
discriminated, base_ok, sem_ok = 0, 0, 0
for k, w in WORLDS.items():
    bt, ba = snapshot_agent(w); st, sa = semantic_agent(w)
    b_hit, s_hit = (ba == EXPECTED[k]), (sa == EXPECTED[k])
    base_ok += b_hit; sem_ok += s_hit
    kill = (not b_hit) and s_hit
    discriminated += kill
    print(f"{k:<14}{'可租' if ba else '不可租':>6}{'可租' if sa else '不可租':>7}{'可租' if EXPECTED[k] else '不可租':>7}"
          f"   {'🔴 杀住基线' if kill else ('——' if s_hit else '❌ 语义Agent挂了')}")
print(f"\n基线正确 {base_ok}/4 vs 语义Agent正确 {sem_ok}/4 | 判别夹具数 = {discriminated}")
print("A103（快照停留在预定期间）与 A104（限制未解除但快照 stale）双双杀住基线 ——")
print("验证体系有牙齿：错的模型在这里翻车，'通过'才有含金量。" if discriminated >= 2 else "⚠️ 判别力不足，夹具需要加强！")

## 4. 七段验证报告生成器：报告是采集出来的，不是写出来的

跑完采集 verdict 矩阵 → 自动统计 L 级覆盖率 → 缺陷账本（能算的算出来，不能算的由审查声明）→ 闸门裁决。

In [ ]:
WEIGHT = {"L1":10, "L2":20, "L3":25, "L4":15, "L5":30}
NEED = {"L1","L2","L3"}   # 必达项

matrix, l5_cov = {}, {}
for k in WORLDS:
    tr, _ = semantic_agent(WORLDS[k])
    matrix[k] = {lv: tr[lv][0] for lv in WEIGHT}
for lv in WEIGHT:
    hits = sum(1 for k in matrix if matrix[k][lv] == "pass")
    applicable = sum(1 for k in matrix if matrix[k][lv] != "n/a")
    l5_cov[lv] = (hits, applicable)

# 缺陷账本：DEF-01 由矩阵统计生成；DEF-02/03 由模型审查声明（报告的诚实性要求）
DEFECTS = []
l5_hits, l5_app = l5_cov["L5"]
DEFECTS.append(("DEF-01", "中", "Capability 层", f"L5 动作建议仅覆盖 1/{l5_app} 夹具（正常流转场景无建议）"))
DEFECTS.append(("DEF-02", "高", "Policy 层", "L4 引用『②层 conditional 授权』为描述而非正式编号——半悬空，装配规则查不到（层内完整性缺口）"))
DEFECTS.append(("DEF-03", "中", "验证设计", "CRE-R-020（进场守卫客户变体）无夹具覆盖——缺陷在验证体系自身，非模型"))

must_ok = all(matrix[k][lv]=="pass" for k in matrix for lv in NEED)
l4_ok = l5_cov["L4"][0] >= 1 and not any(d[1]=="高" and d[0]=="DEF-02" and False for d in DEFECTS)
gate = "PASS WITH CONDITIONS" if (must_ok and any(d[1]=="高" for d in DEFECTS)) else ("PASS" if must_ok else "FAIL")

report = f"""
╔══════════════════════════════════════════════════════════════════╗
║        MI CRE 数字员工验证报告 v0.1 · 闸门裁决：{gate:<22}║
╠══════════════════════════════════════════════════════════════════╣
║ 1 范围      夹具 A101-A104（铺位可租性）· 判定梯 L1→L5 v1        ║
║            对照：快照式基线 Agent · 盲测：A104                     ║
║ 2 判定结果  {'必达项 L1-L3 全绿 ✅' if must_ok else '必达项有挂 ❌'}                         ║"""
report += "\n║            " + "  ".join(f"{lv}:{h}/{a}" for lv,(h,a) in l5_cov.items()) + " (命中/适用)" + " "*6 + "║\n"
report += f"""║ 3 对照证据  基线 {base_ok}/4 vs 语义Agent {sem_ok}/4 · 判别夹具 {discriminated} 个        ║
║            A103/A104 杀住快照式实现（验证有牙齿）                  ║
║ 4 缺陷账本                                                      ║"""
for did, sev, layer, desc in DEFECTS:
    report += f"\n║            [{did}|{sev}] {layer}：{desc[:36]:<38}║"
report += f"""
║ 5 覆盖声明  未测：并发预定竞争 / 跨项目 / CRE-R-020 客户变体 /    ║
║            审批流 K1-K3 全链 —— 结论作用域 ≤ 夹具作用域            ║
║ 6 闸门裁决  {gate}                                    ║
║            放行条件：DEF-02 修复（Policy 层补编号）后 L4 复验      ║
║ 7 v0.2 待办 DEF-01/02/03 → v0.2 backlog → 再验证（语义模型 CI）   ║
╚══════════════════════════════════════════════════════════════════╝"""
print(report)

## 5. 可视化：判别力 × 判定梯覆盖

In [ ]:
import numpy as np
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

# 左图：对照赛——每夹具基线 vs 语义Agent（1=正确 0=错误）
names = list(WORLDS.keys())
base_scores = [float(snapshot_agent(WORLDS[k])[1] == EXPECTED[k]) for k in names]
sem_scores  = [float(semantic_agent(WORLDS[k])[1]  == EXPECTED[k]) for k in names]
x = np.arange(len(names)); wdt = 0.36
ax1.bar(x-wdt/2, base_scores, wdt, color="#c0504d", label="快照式基线（信任落库字段）")
ax1.bar(x+wdt/2, sem_scores, wdt, color="#4f81bd", label="语义模型 Agent（从事实重算）")
for i,(b,s) in enumerate(zip(base_scores, sem_scores)):
    ax1.text(i-wdt/2, b+0.02, "对" if b else "错", ha="center", fontsize=11)
    ax1.text(i+wdt/2, s+0.02, "对" if s else "错", ha="center", fontsize=11)
ax1.set_xticks(x); ax1.set_xticklabels([n.split()[0]+"\n"+n.split()[1] for n in names], fontsize=10)
ax1.set_yticks([0,1]); ax1.set_yticklabels(["答错","答对"]); ax1.set_ylim(0,1.15)
ax1.set_title(f"对照赛：判别力 = 基线挂而语义Agent过（杀住 {discriminated} 个）")
ax1.legend(loc="lower right", fontsize=9); ax1.grid(axis="y", alpha=0.3)

# 右图：判定梯 L1→L5 覆盖（命中/适用）+ 必达线
lvs = list(WEIGHT.keys())
hits = [l5_cov[l][0] for l in lvs]; apps = [l5_cov[l][1] for l in lvs]
colors = ["#4f81bd" if l in NEED else "#9bbb59" for l in lvs]
ax2.bar(lvs, hits, color=colors, label="命中（pass）")
ax2.bar(lvs, [a-h for a,h in zip(apps,hits)], bottom=hits, color="#d9d9d9", label="适用未命中")
for i,(h,a) in enumerate(zip(hits,apps)):
    ax2.text(i, a+0.05, f"{h}/{a}", ha="center", fontsize=11)
ax2.axhline(y=1, color="#c0504d", ls="--", lw=1.5)
ax2.text(3.55, 1.05, "必达地板：L1-L3 全绿（必达）", color="#c0504d", fontsize=9)
ax2.set_ylim(0,4.6); ax2.set_title(f"判定梯覆盖 · 闸门：{gate}")
ax2.legend(loc="upper left", fontsize=9); ax2.grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.savefig("d7_validation_gate.png", dpi=120, bbox_inches="tight"); plt.show()
print("已保存 d7_validation_gate.png")

## 收束：四周轨道的最后一条认知

- **基线对照**证明验证有牙齿（A103/A104 杀住快照式实现）；
- **盲测 A104** 证明声明式规则的泛化力（谓词求值，不是为夹具定制的 if-else）；
- **七段报告**把"通过"变成证据集 + 缺陷账本 + 覆盖声明 + 三级闸门裁决——`PASS WITH CONDITIONS` 带账上线、按账迭代。

四周闭环合拢：`v0.1 模型 → 验证 → 报告（证据+缺陷）→ v0.2 backlog → 再验证`——语义模型的 CI 流水线。验证报告就是这条流水线的 CI 产物，而 defect 账本里最诚实的一条，永远是对验证体系自身的指控（DEF-03）。